# Four sources: five-fold CV and prediction ensembling

Four-source model with fixed splits and checkpoint selection by validation MSE.
Each fold uses four partitions for training and one for validation.
**Test aggregation follows DrugForm-DTA**: all five models predict the same test samples;
average predictions per sample, then compute MSE, RMSE, CI (concordance index), and Pearson.
No additional 90/10 validation split or UniQSAR checkpoint-weight averaging is applied.

Inputs: `train.pt`, `test.pt` in v4 format and `davis_folds.json`/`kiba_folds.json`.
This standalone notebook needs no repository on Kaggle. Ensemble metrics are withheld
until all folds are available; the mean of per-fold metrics is not the primary result.

Outputs: `ensemble_predictions.csv` contains true labels, per-fold predictions, and their mean;
`ensemble_metrics.json`/`.csv` contains ensemble and individual-model metrics for comparison;
`cv_summary.json` records whether all five folds are complete. Undefined metrics are stored as null.
Regenerate both notebooks: `python notebook/sync_ensemble_notebooks.py`.


In [ ]:
from pathlib import Path

DATASET = 'davis'  # 'davis' or 'kiba'
DATA_DIR = None  # Path('/kaggle/input/.../davis') containing both train.pt and test.pt.
REPO_ROOT = next((p for base in (Path.cwd(), *Path.cwd().parents)
                  for p in (base, base / 'DATN_DTA')
                  if (p / 'training.py').is_file() and (p / 'models').is_dir()), Path.cwd())
if DATASET not in ('davis', 'kiba'):
    raise ValueError('DATASET must be davis or kiba')
if DATA_DIR is None:
    if Path('/kaggle/input').is_dir():
        candidates = sorted({p.parent for p in Path('/kaggle/input').rglob('train.pt')
                             if (p.parent / 'test.pt').is_file()})
        matching = [p for p in candidates if DATASET in p.as_posix().lower()]
        candidates = matching or candidates
        if len(candidates) != 1:
            raise ValueError(f'Found {len(candidates)} datasets; set DATA_DIR explicitly: {candidates}')
        DATA_DIR = candidates[0]
    else:
        DATA_DIR = REPO_ROOT / 'data/four_source_embeddings' / DATASET
DATA_DIR = Path(DATA_DIR)
OUTPUT_DIR = (Path('/kaggle/working') if Path('/kaggle/working').is_dir()
              else REPO_ROOT / 'outputs') / f'{DATASET}_four_source_cv_ensemble'
INPUT_DIR = Path('/kaggle/input')
FOLD = None  # None: run all five folds sequentially; 0..4: run one fold.
CHECK_FOLDS_ONLY = False  # True: validate data/folds on CPU without training.
TRAIN_BATCH_SIZE = 512  # Global batch, split across GPUs when MAX_GPUS > 1.
TEST_BATCH_SIZE = 512
MAX_GPUS = 2
CUDA_DEVICE = 'cuda:0'
LR = 0.0005  # Fixed throughout training.
NUM_EPOCHS = 1000  # Per fold, including epochs completed before resuming.
CHECKPOINT_EVERY = 100
AUTO_RESUME = True
RESUME_PATH = None  # Full checkpoint path; requires FOLD=0..4 when specified.
SEED = 42
SHUFFLE_TRAIN = False
LOG_INTERVAL = 20
MODEL_CONFIG = {'num_features_xd': 78, 'num_features_xt': 41, 'n_output': 1,
                'output_dim': 128, 'dropout': 0.2, 'protein_embedding_dim': 640}
print('Dataset:', DATASET, '| data:', DATA_DIR, '| output:', OUTPUT_DIR)

FOLDS_PATH = None  # Path('/kaggle/input/.../davis_folds.json'); None: auto-discover.


In [ ]:
import importlib.util
import subprocess
import sys

if not CHECK_FOLDS_ONLY and importlib.util.find_spec('torch_geometric') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch-geometric'])


In [ ]:
"""Four-source entity tables, indexed batches and portable training checkpoints.

This module does not import PyG, create a model, or run training on import.
Graph/embedding banks are read-only ordinary Python objects, not model buffers.
"""
from pathlib import Path
from types import SimpleNamespace
import csv
import hashlib
import json
import random

import numpy as np
import torch
import torch.nn as nn

class FourSourceSplit:
    """Load entities once; after GPU upload, retain indices, labels, and IDs for CSV export."""
    def __init__(self, path, expected_split, dataset=None, protein_embedding_dim=640):
        self.data = torch.load(path, map_location='cpu', weights_only=True, mmap=True)
        if self.data.get('format_version') != 4:
            raise ValueError(f'{path}: expected a v4 split file generated by create_data.py')
        self.dataset = self.data.get('dataset')
        if (self.dataset not in ('davis', 'kiba') or self.data.get('split') != expected_split
                or (dataset is not None and self.dataset != dataset)):
            raise ValueError(f'{path}: dataset/split does not match {dataset or "davis/kiba"} {expected_split}')
        self.ligand_ids = self.data['ligand_ids']
        self.protein_ids = self.data['protein_ids']
        self.pairs = self.data['pairs']
        self.labels = self.pairs['y'].float().reshape(-1)
        n = len(self.labels)
        if (self.pairs['y'].ndim != 1 or n == 0 or len(self.pairs['ligand_index']) != n
                or len(self.pairs['protein_index']) != n or len(self.pairs['row_id']) != n
                or not torch.isfinite(self.labels).all()):
            raise ValueError(f'{path}: invalid pair or label tables')
        nl, np_ = len(self.data['ligand_ids']), len(self.data['protein_ids'])
        if (len(self.data['ligand_graphs']) != nl or len(self.data['ligand_embeddings']) != nl
                or len(self.data['protein_graphs']) != np_
                or len(self.data['protein_embeddings']) != np_
                or len(self.data['protein_embedding_ids']) != np_
                or len(set(self.ligand_ids)) != nl or len(set(self.protein_ids)) != np_):
            raise ValueError(f'{path}: four-source table dimensions do not match')
        for key, size in [('ligand_index', nl), ('protein_index', np_)]:
            indices = self.pairs[key]
            if (indices.dtype != torch.long or indices.shape != (n,)
                    or indices.min() < 0 or indices.max() >= size):
                raise ValueError(f'{path}: invalid {key} indices')
        # Both sources contain unpadded token matrices; normalize dtype once per entity.
        for key, width in [('ligand_embeddings', 512),
                           ('protein_embeddings', protein_embedding_dim)]:
            for i, value in enumerate(self.data[key]):
                if (not isinstance(value, torch.Tensor) or value.ndim != 2
                        or value.shape[0] == 0 or value.shape[1] != width
                        or not value.is_floating_point() or not torch.isfinite(value).all()):
                    raise ValueError(f'{path}: {key}[{i}] must be finite with shape [L>0,{width}]')
                self.data[key][i] = value.detach().float()

    def __len__(self):
        return len(self.labels)

    def pair_ids(self, index):
        li = int(self.pairs['ligand_index'][index])
        pi = int(self.pairs['protein_index'][index])
        return self.ligand_ids[li], self.protein_ids[pi]


def pack_segments(values):
    """Concatenate segments once on CPU and store offsets without padding or pooling tokens."""
    lengths = torch.tensor([value.shape[0] for value in values], dtype=torch.long)
    ptr = torch.cat((torch.zeros(1, dtype=torch.long), lengths.cumsum(0)))
    return torch.cat(values, dim=0).contiguous(), ptr


def pack_graphs(graphs, width):
    for graph in graphs:
        x, edges = graph['x'], graph['edge_index']
        if (x.ndim != 2 or x.shape[0] == 0 or x.shape[1] != width
                or not torch.isfinite(x).all() or edges.ndim != 2 or edges.shape[0] != 2
                or edges.dtype != torch.long
                or (edges.numel() and (edges.min() < 0 or edges.max() >= len(x)))):
            raise ValueError(f'Graph requires finite x with shape [N>0,{width}] and valid edge_index')
    x, node_ptr = pack_segments([graph['x'].detach().float() for graph in graphs])
    # Edges store LOCAL node indices; add offsets only when selecting graphs for a batch.
    edges, edge_ptr = pack_segments([graph['edge_index'].t() for graph in graphs])
    return {'x': x, 'node_ptr': node_ptr, 'edges': edges, 'edge_ptr': edge_ptr}


def same_graph(a, b):
    return torch.equal(a['x'], b['x']) and torch.equal(a['edge_index'], b['edge_index'])


def build_cpu_tables(splits):
    """Merge train/test entities; use original graph IDs and sequence-hash protein embedding IDs."""
    if len({split.dataset for split in splits}) != 1:
        raise ValueError('DAVIS and KIBA cannot share one bank')
    ligands, proteins, protein_embeddings = {}, {}, {}
    for split in splits:
        data = split.data
        for i, lid in enumerate(split.ligand_ids):
            graph, embedding = data['ligand_graphs'][i], data['ligand_embeddings'][i]
            if lid in ligands:
                old_graph, old_embedding = ligands[lid]
                if not same_graph(old_graph, graph) or not torch.equal(old_embedding, embedding):
                    raise ValueError(f'Ligand {lid}: train/test data differ')
            else:
                ligands[lid] = (graph, embedding)
        for i, pid in enumerate(split.protein_ids):
            graph = data['protein_graphs'][i]
            eid = data['protein_embedding_ids'][i]
            embedding = data['protein_embeddings'][i]
            if pid in proteins:
                old_graph, old_eid = proteins[pid]
                if old_eid != eid or not same_graph(old_graph, graph):
                    raise ValueError(f'Protein {pid}: train/test data differ')
            else:
                proteins[pid] = (graph, eid)
            if eid in protein_embeddings:
                if not torch.equal(protein_embeddings[eid], embedding):
                    raise ValueError(f'Embedding {eid}: same ID has different values')
            else:
                protein_embeddings[eid] = embedding
    ligand_ids, protein_ids, embedding_ids = sorted(ligands), sorted(proteins), sorted(protein_embeddings)
    ligand_map = {key: i for i, key in enumerate(ligand_ids)}
    protein_map = {key: i for i, key in enumerate(protein_ids)}
    embedding_map = {key: i for i, key in enumerate(embedding_ids)}
    tables = {}
    for prefix, graphs, width in (
            ('ligand', [ligands[key][0] for key in ligand_ids], 78),
            ('protein', [proteins[key][0] for key in protein_ids], 41)):
        for key, value in pack_graphs(graphs, width).items():
            tables[f'{prefix}_{key}'] = value
    tables['ligand_embedding'], tables['ligand_embedding_ptr'] = pack_segments(
        [ligands[key][1] for key in ligand_ids])
    tables['protein_embedding'], tables['protein_embedding_ptr'] = pack_segments(
        [protein_embeddings[key] for key in embedding_ids])
    tables['protein_to_embedding'] = torch.tensor(
        [embedding_map[proteins[key][1]] for key in protein_ids], dtype=torch.long)
    return tables, ligand_map, protein_map


def execution_devices(device, use_data_parallel=True, max_gpus=2):
    """Put the primary GPU first; default to at most two GPUs, with single-GPU/CPU fallback."""
    device = torch.device(device)
    if max_gpus < 1:
        raise ValueError('MAX_GPUS must be >= 1')
    if device.type != 'cuda':
        return [device]
    count = torch.cuda.device_count()
    primary = device.index if device.index is not None else torch.cuda.current_device()
    if not 0 <= primary < count:
        raise ValueError(f'Primary GPU cuda:{primary} is not among the {count} available GPUs')
    ids = [primary]
    if use_data_parallel:
        ids += [i for i in range(count) if i != primary]
    return [torch.device('cuda', i) for i in ids[:max_gpus]]


def take_segments(values, ptr, indices):
    """Look up segments and build batch vectors on the bank's device (GPU during training)."""
    starts = ptr.index_select(0, indices)
    lengths = ptr.index_select(0, indices + 1) - starts
    batch_ptr = torch.cat((lengths.new_zeros(1), lengths.cumsum(0)))
    owners = torch.repeat_interleave(torch.arange(indices.numel(), device=indices.device), lengths)
    local = torch.arange(owners.numel(), device=indices.device) - batch_ptr.index_select(0, owners)
    positions = starts.index_select(0, owners) + local
    return values.index_select(0, positions), owners, batch_ptr


class DeviceFourSourceStore:
    """Each GPU holds a tensor dictionary; replicas share a read-only Python store."""
    def __init__(self, cpu_tables, devices, ligand_map, protein_map):
        self.devices = tuple(torch.device(device) for device in devices)
        if not self.devices or len(set(self.devices)) != len(self.devices):
            raise ValueError('Expected a non-empty list of distinct devices')
        self.device = self.devices[0]  # Pair indices and labels reside on the primary GPU.
        self.ligand_id_to_index = ligand_map
        self.protein_id_to_index = protein_map
        self.nbytes = sum(value.numel() * value.element_size() for value in cpu_tables.values())
        self.tables = {}
        for target in self.devices:
            if target.type == 'cuda':
                free, _ = torch.cuda.mem_get_info(target)
                if self.nbytes > free:
                    raise RuntimeError(f'Bank on {target} requires {self.nbytes / 1024**3:.3f} GiB, '
                                       f'but only {free / 1024**3:.3f} GiB is free. Restart the session to release the old bank.')
            # Upload once per GPU. The bank is not a model buffer, so DataParallel
            # does not broadcast/copy it on each forward pass, and checkpoints exclude it.
            self.tables[str(target)] = {
                key: value.detach().to(target) for key, value in cpu_tables.items()}
            print(f'Four sources on {target}: {self.nbytes / 1024**3:.3f} GiB; '
                  f'{len(ligand_map)} ligand graphs, {len(protein_map)} protein graphs, '
                  f'{cpu_tables["protein_embedding_ptr"].numel() - 1} protein embeddings')

    def on(self, device):
        key = str(torch.device(device))
        if key not in self.tables:
            raise ValueError(f'Bank is not prepared on {key}')
        return self.tables[key]

    def graph_batch(self, prefix, indices):
        t = self.on(indices.device)
        x, batch, node_ptr = take_segments(t[f'{prefix}_x'], t[f'{prefix}_node_ptr'], indices)
        edges, edge_batch, _ = take_segments(t[f'{prefix}_edges'], t[f'{prefix}_edge_ptr'], indices)
        edge_index = (edges + node_ptr.index_select(0, edge_batch).unsqueeze(1)).t().contiguous()
        return x, edge_index, batch

    def lookup(self, pair_indices):
        # DataParallel splits INDICES; each replica looks up its local batch on the same device.
        t = self.on(pair_indices.device)
        li, pi = pair_indices[:, 0], pair_indices[:, 1]
        drug_x, drug_edges, drug_batch = self.graph_batch('ligand', li)
        prot_x, prot_edges, prot_batch = self.graph_batch('protein', pi)
        drug_embedding, drug_embedding_batch, _ = take_segments(
            t['ligand_embedding'], t['ligand_embedding_ptr'], li)
        embedding_indices = t['protein_to_embedding'].index_select(0, pi)
        protein_embedding, protein_embedding_batch, _ = take_segments(
            t['protein_embedding'], t['protein_embedding_ptr'], embedding_indices)
        # Tensors already reside on the GPU; wrap attributes without CPU collation or copying.
        return (SimpleNamespace(x=drug_x, edge_index=drug_edges, batch=drug_batch,
                     drug_embedding=drug_embedding, drug_embedding_batch=drug_embedding_batch),
                SimpleNamespace(x=prot_x, edge_index=prot_edges, batch=prot_batch,
                     protein_embedding=protein_embedding, protein_embedding_batch=protein_embedding_batch))


class IndexBatches:
    """Yield only GPU pair indices and labels per batch; do not fetch graphs/tokens on CPU."""
    def __init__(self, split, store, batch_size, shuffle=False, seed=42):
        if batch_size < 1:
            raise ValueError('Batch size must be >= 1')
        self.size, self.batch_size = len(split), batch_size
        self.shuffle, self.device = shuffle, store.device
        self.generator = torch.Generator(device=self.device).manual_seed(seed)
        ligand_remap = torch.tensor([store.ligand_id_to_index[key] for key in split.ligand_ids])
        protein_remap = torch.tensor([store.protein_id_to_index[key] for key in split.protein_ids])
        self.pairs = torch.stack((
            ligand_remap.index_select(0, split.pairs['ligand_index']),
            protein_remap.index_select(0, split.pairs['protein_index'])), dim=1).to(self.device)
        self.labels = split.labels.reshape(-1, 1).to(self.device)

    def __len__(self):
        return (self.size + self.batch_size - 1) // self.batch_size

    def reset_seed(self, seed):
        self.generator.manual_seed(seed)

    def __iter__(self):
        order = (torch.randperm(self.size, generator=self.generator, device=self.device)
                 if self.shuffle else torch.arange(self.size, device=self.device))
        for start in range(0, self.size, self.batch_size):
            rows = order[start:start + self.batch_size]
            yield self.pairs.index_select(0, rows), self.labels.index_select(0, rows)


def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def save_checkpoint(checkpoint, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    try:
        torch.save(checkpoint, temporary)
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)

def save_test_predictions(path, dataset, labels, predictions):
    if not len(dataset) == len(labels) == len(predictions):
        raise ValueError('Test sample and prediction counts do not match')
    with Path(path).open('w', newline='', encoding='utf-8') as handle:
        writer = csv.writer(handle)
        writer.writerow(['row_id', 'ligand_id', 'protein_id', 'affinity', 'predicted_affinity'])
        for i, (label, prediction) in enumerate(zip(labels, predictions)):
            ligand_id, protein_id = dataset.pair_ids(i)
            writer.writerow([dataset.pairs['row_id'][i], ligand_id, protein_id,
                             float(label), float(prediction)])

def cpu_snapshot(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: cpu_snapshot(item) for key, item in value.items()}
    if isinstance(value, list):
        return [cpu_snapshot(item) for item in value]
    if isinstance(value, tuple):
        return tuple(cpu_snapshot(item) for item in value)
    return value

def dataset_signature(splits):
    # Validate pair/label order and IDs without rereading large embedding tensors.
    digest = hashlib.sha256()
    for split in splits:
        digest.update(json.dumps([split.ligand_ids, split.protein_ids],
                                 ensure_ascii=False).encode('utf-8'))
        for key in ('ligand_index', 'protein_index', 'y'):
            value = split.pairs[key].detach().cpu().contiguous()
            digest.update(str((key, tuple(value.shape), str(value.dtype))).encode())
            digest.update(value.numpy().tobytes())
    return digest.hexdigest()

def model_signature(model):
    core = unwrap_model(model)
    description = {
        'modules': [(name, ('__main__' if name == '' else type(module).__module__), type(module).__qualname__, repr(module))
                    for name, module in core.named_modules()],
        'state': [(key, list(value.shape), str(value.dtype))
                  for key, value in core.state_dict().items()],
    }
    return hashlib.sha256(json.dumps(description, sort_keys=True).encode()).hexdigest()

def capture_rng(batches, devices):
    numpy_state = np.random.get_state()
    return {
        'python': random.getstate(),
        'numpy': {'name': numpy_state[0], 'keys': numpy_state[1].tolist(),
                  'position': numpy_state[2], 'has_gauss': numpy_state[3],
                  'cached_gaussian': numpy_state[4]},
        'torch_cpu': torch.get_rng_state().clone(),
        'torch_cuda': [torch.cuda.get_rng_state(target).cpu()
                       for target in devices if target.type == 'cuda'],
        'batch_generator': batches.generator.get_state().cpu(),
        'batch_generator_device': str(batches.generator.device),
    }

def restore_rng(state, batches, devices):
    random.setstate(state['python'])
    ns = state['numpy']
    np.random.set_state((ns['name'], np.array(ns['keys'], dtype=np.uint32),
                         ns['position'], ns['has_gauss'], ns['cached_gaussian']))
    torch.set_rng_state(state['torch_cpu'])
    targets = [target for target in devices if target.type == 'cuda']
    if len(targets) == len(state['torch_cuda']):
        for target, value in zip(targets, state['torch_cuda']):
            torch.cuda.set_rng_state(value, target)
    else:
        print('GPU count changed: keeping the new session CUDA seed; exact replay is unavailable.')
    saved_type = torch.device(state['batch_generator_device']).type
    if saved_type == torch.device(batches.generator.device).type:
        batches.generator.set_state(state['batch_generator'])
    else:
        print('CPU/CUDA generator changed: keeping the new session shuffle seed.')

def resume_problem(payload, expected):
    if not isinstance(payload, dict) or payload.get('checkpoint_kind') != 'training_resume_v1':
        return 'not a full resume checkpoint (old best/weights file lacks optimizer/RNG state)'
    required = ('model_state_dict', 'optimizer_state_dict', 'epoch', 'best_checkpoint',
                'history', 'rng_state', 'best_predictions')
    if any(key not in payload for key in required):
        return 'missing training state'
    for key, value in expected.items():
        if payload.get(key) != value:
            return f'{key} differs from the current run'
    epoch = payload['epoch']
    if not isinstance(epoch, int) or epoch < 1:
        return 'invalid epoch'
    if [row.get('epoch') for row in payload['history']] != list(range(1, epoch + 1)):
        return 'history does not match epoch'
    best = payload['best_checkpoint']
    score_key = 'validation_mse' if isinstance(best, dict) and best.get('selection_split') == 'validation' else 'test_mse'
    if (not isinstance(best, dict) or 'model_state_dict' not in best
            or not 1 <= best.get('epoch', 0) <= epoch
            or not np.isfinite(best.get(score_key, float('nan')))):
        return 'invalid best checkpoint'
    return None

def find_resume_checkpoint(roots, expected, explicit_path=None):
    if explicit_path is not None:
        candidates = [Path(explicit_path)]
        if not candidates[0].is_file():
            raise FileNotFoundError(f'RESUME_PATH not found: {explicit_path}')
    else:
        candidates = set()
        for root in roots:
            root = Path(root)
            if root.is_dir():
                for pattern in ('*checkpoint*.pt', '*checkpoint*.pth', 'best_model.pt',
                                'best_until_epoch_*.pt'):
                    candidates.update(root.rglob(pattern))
        candidates = sorted(candidates, key=lambda p: str(p))
    selected, selected_path = None, None
    for path in candidates:
        try:
            # New resume files use zip-based torch.save; mmap does not upload weights to the GPU.
            payload = torch.load(path, map_location='cpu', weights_only=True, mmap=True)
            problem = resume_problem(payload, expected)
        except Exception as exc:
            payload, problem = None, f'cannot read: {exc}'
        if problem:
            if explicit_path is not None:
                raise ValueError(f'Cannot resume {path}: {problem}')
            print(f'Skipping {path.name}: {problem}')
            continue
        if selected is None or payload['epoch'] > selected['epoch']:
            selected, selected_path = payload, path
    return selected_path, selected

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_data(data_dir, dataset, device, batch_size=512, test_batch_size=512,
              max_gpus=2, shuffle=False, seed=42):
    """Validate splits, upload one bank per device, then release CPU entity lists."""
    data_dir = Path(data_dir)
    train_data = FourSourceSplit(data_dir / 'train.pt', 'train', dataset)
    test_data = FourSourceSplit(data_dir / 'test.pt', 'test', dataset)
    cpu_tables, ligand_map, protein_map = build_cpu_tables((train_data, test_data))
    devices = execution_devices(device, use_data_parallel=max_gpus > 1, max_gpus=max_gpus)
    store = DeviceFourSourceStore(cpu_tables, devices, ligand_map, protein_map)
    train_batches = IndexBatches(train_data, store, batch_size, shuffle, seed)
    test_batches = IndexBatches(test_data, store, test_batch_size, False, seed)
    train_data.data = test_data.data = None
    print(f'{dataset.upper()}: {len(train_data):,} train / {len(test_data):,} test')
    print('Devices:', [str(d) for d in devices], '| global batch:', batch_size)
    return train_data, test_data, store, train_batches, test_batches


In [ ]:
if not CHECK_FOLDS_ONLY:
    """Four-source DTA model; matches the current original-copy notebook."""
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch_geometric.nn import GATConv, GCNConv
    from torch_geometric.nn import global_mean_pool as gap
    from torch_geometric.nn import global_max_pool as gmp
    # GAT  model
    class GATnGCN_Model(torch.nn.Module):
        def __init__(self, data_store, num_features_xd=78, n_output=1, num_features_xt=41,
                          output_dim=128, dropout=0.2, protein_embedding_dim=640):
            super(GATnGCN_Model, self).__init__()
            # The store is a plain Python object; state_dict contains only model weights.
            self.data_store = data_store
            if output_dim != 128:
                raise ValueError('Fusion requires output_dim=128')
            # Ligand graph: GAT 8x64 = 512 -> GAT 4x64 = 256; max+mean pooling = 512.
            self.conv1 = GATConv(num_features_xd, 64, heads=8, concat=True)
            self.conv2 = GATConv(512, 64, heads=4, concat=True)
            self.fc_g1 = torch.nn.Linear(512, 256)
            self.fc_g2 = torch.nn.Linear(256, output_dim)

            # Replace ECFP with Chemformer ligand embeddings. Pool only in forward.
            self.fc_drug_embedding = nn.Linear(2 * 512, 128)
            self.bn_drug_embedding = nn.BatchNorm1d(128)
            # Protein graph: GAT 8x64 -> GCN 256, with 41 input features and separate weights.
            self.conv_prot1 = GATConv(num_features_xt, 64, heads=8, concat=True)
            self.conv_prot2 = GCNConv(512, 256)
            self.fc_prot_g1 = nn.Linear(512, 256)
            self.fc_prot_g2 = nn.Linear(256, output_dim)

            # Protein ESM-2: concatenate max/mean pooling, then project 1280 -> 128.
            self.fc_protein_embedding = nn.Linear(2 * protein_embedding_dim, 128)
            self.bn_protein_embedding = nn.BatchNorm1d(128)

            #concat
            self.fc_xd = nn.Linear(256,128)
            self.fc_xp = nn.Linear(256,128)

            # Small prediction head: 256 -> 512 -> 128 -> affinity.
            self.fc1 = nn.Linear(256, 512)
            self.fc2 = nn.Linear(512, 128)
            self.out = nn.Linear(128, n_output)

            # activation and regularization
            self.relu = nn.ReLU()
            self.dropout = nn.Dropout(dropout)

        def forward(self, pair_indices):
            # Look up all four sources on the GPU at the start of each forward pass.
            data_drug, data_protein = self.data_store.lookup(pair_indices)
            # graph input feed-forward
            x_drug, edge_index_drug, batch_drug = data_drug.x, data_drug.edge_index, data_drug.batch

            x_drug = self.conv1(x_drug, edge_index_drug)
            x_drug = self.relu(x_drug)
            x_drug = self.conv2(x_drug, edge_index_drug)
            x_drug = self.relu(x_drug)
            # apply global max pooling (gmp) and global mean pooling (gap)
            x_drug = torch.cat([gmp(x_drug, batch_drug), gap(x_drug, batch_drug)], dim=1)
            x_drug = self.relu(self.fc_g1(x_drug))
            x_drug = self.dropout(x_drug)
            x_drug = self.fc_g2(x_drug)

            # Tokens are concatenated; drug_embedding_batch identifies their samples.
            # The bank preserves raw tokens; max+mean pooling happens only here.
            # Concatenate [max, mean], pooling each sample separately: [B,1024].
            x_drug_embedding = torch.cat([
                gmp(data_drug.drug_embedding, data_drug.drug_embedding_batch),
                gap(data_drug.drug_embedding, data_drug.drug_embedding_batch),
            ], dim=1)
            x_drug_embedding = self.fc_drug_embedding(x_drug_embedding)
            # BatchNorm needs >=2 training samples; use running statistics for a singleton batch.
            if self.training and x_drug_embedding.size(0) == 1:
                x_drug_embedding = F.batch_norm(
                    x_drug_embedding, self.bn_drug_embedding.running_mean,
                    self.bn_drug_embedding.running_var, self.bn_drug_embedding.weight,
                    self.bn_drug_embedding.bias, training=False, eps=self.bn_drug_embedding.eps)
            else:
                x_drug_embedding = self.bn_drug_embedding(x_drug_embedding)
            x_drug_embedding = self.relu(x_drug_embedding)
            x_drug_embedding = self.dropout(x_drug_embedding)

            # protein input feed-forward:
            x_prots, edge_index_prots, prots_batch= data_protein.x, data_protein.edge_index, data_protein.batch
            # Protein: GAT -> GCN -> max+mean -> Linear/ReLU/Dropout -> Linear.
            x_prots = self.conv_prot1(x_prots, edge_index_prots)
            x_prots = self.relu(x_prots)
            x_prots = self.conv_prot2(x_prots, edge_index_prots)
            x_prots = self.relu(x_prots)
            x_prots = torch.cat([gmp(x_prots, prots_batch), gap(x_prots, prots_batch)], dim=1)
            x_prots = self.relu(self.fc_prot_g1(x_prots))
            x_prots = self.dropout(x_prots)
            x_prots = self.fc_prot_g2(x_prots)
            # Pool tokens per sample using protein_embedding_batch, not the graph node batch.
            # Both pooling operations use each sample's actual tokens without padding.
            # Concatenation order: [max (640), mean (640)] -> [B,1280].
            x_protein_embedding = torch.cat([
                gmp(data_protein.protein_embedding, data_protein.protein_embedding_batch),
                gap(data_protein.protein_embedding, data_protein.protein_embedding_batch),
            ], dim=1)
            x_protein_embedding = self.fc_protein_embedding(x_protein_embedding)
            if self.training and x_protein_embedding.size(0) == 1:
                x_protein_embedding = F.batch_norm(
                    x_protein_embedding, self.bn_protein_embedding.running_mean,
                    self.bn_protein_embedding.running_var, self.bn_protein_embedding.weight,
                    self.bn_protein_embedding.bias, training=False, eps=self.bn_protein_embedding.eps)
            else:
                x_protein_embedding = self.bn_protein_embedding(x_protein_embedding)
            x_protein_embedding = self.relu(x_protein_embedding)
            x_protein_embedding = self.dropout(x_protein_embedding)

            # Ablation: concatenate graph features directly, without cross-attention or residuals.
            xd = torch.cat((x_drug, x_drug_embedding), dim=1)       # [B,256]
            xp = torch.cat((x_prots, x_protein_embedding), dim=1)  # [B,256]
            xd = self.relu(self.fc_xd(xd))
            xp = self.relu(self.fc_xp(xp))

            xc = torch.cat((xd,xp), 1)
            # Apply ReLU/Dropout after each hidden layer; the output layer is linear.
            xc = self.fc1(xc)
            xc = self.relu(xc)
            xc = self.dropout(xc)
            xc = self.fc2(xc)
            xc = self.relu(xc)
            xc = self.dropout(xc)
            out = self.out(xc)
            return out

In [ ]:
"""Train the four-source DTA model with fixed LR and resumable checkpoints."""
from __future__ import annotations

import argparse
import csv
from dataclasses import dataclass, field
from pathlib import Path
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


ROOT = REPO_ROOT
MODEL_TYPE = 'GATnGCN_DrugGAT8x64_GAT4x64_ProteinGATGCN_DualEmbeddingMaxMean_SmallMLP'
DEFAULT_MODEL_CONFIG = {'num_features_xd': 78, 'num_features_xt': 41, 'n_output': 1,
                        'output_dim': 128, 'dropout': 0.2, 'protein_embedding_dim': 640}


@dataclass
class TrainingConfig:
    dataset: str = 'davis'
    data_dir: Path | None = None
    output_dir: Path | None = None
    device: str = 'cuda:0'
    max_gpus: int = 2
    batch_size: int = 512
    test_batch_size: int = 512
    lr: float = 0.0005
    epochs: int = 1000
    checkpoint_every: int = 100
    auto_resume: bool = True
    resume_path: Path | None = None
    input_dir: Path = Path('/kaggle/input')
    shuffle: bool = False
    seed: int = 42
    log_interval: int = 20
    model_config: dict = field(default_factory=lambda: dict(DEFAULT_MODEL_CONFIG))

    def __post_init__(self):
        if self.dataset not in ('davis', 'kiba'):
            raise ValueError('dataset must be davis or kiba')
        if min(self.batch_size, self.test_batch_size, self.max_gpus, self.epochs,
               self.checkpoint_every) < 1 or self.lr <= 0 or not np.isfinite(self.lr):
            raise ValueError('Batch sizes, epochs, max_gpus, checkpoint_every and LR must be positive')
        if self.log_interval < 0:
            raise ValueError('log_interval must be >= 0')
        self.data_dir = Path(self.data_dir) if self.data_dir else ROOT / 'data/four_source_embeddings' / self.dataset
        base = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else ROOT / 'outputs'
        self.output_dir = Path(self.output_dir) if self.output_dir else base / f'{self.dataset}_drug_gat8x64_gat4x64_dual_embedding_maxmean'
        self.input_dir = Path(self.input_dir)
        self.resume_path = Path(self.resume_path) if self.resume_path else None


def configure_model(model_config, store):

    for target in store.devices:
        store.on(target)
    core = GATnGCN_Model(data_store=store, **model_config).to(store.device)
    if len(store.devices) > 1:
        gpu_ids = [target.index for target in store.devices]
        return nn.DataParallel(core, device_ids=gpu_ids, output_device=gpu_ids[0], dim=0)
    return core

def train(model, batches, optimizer, epoch, log_interval=20):
    model.train()
    total_loss, count = 0.0, 0
    for batch_idx, (pair_indices, labels) in enumerate(batches):
        optimizer.zero_grad(set_to_none=True)
        output = model(pair_indices)
        # Optimize MSE over the full batch gathered from all GPUs.
        loss = F.mse_loss(output, labels)
        value = loss.item()
        if not np.isfinite(value):
            raise FloatingPointError(f'Loss contains NaN/Inf at epoch {epoch}, batch {batch_idx + 1}')
        loss.backward()
        optimizer.step()
        size = pair_indices.size(0)
        total_loss += value * size
        count += size
        if log_interval and (batch_idx == 0 or (batch_idx + 1) % log_interval == 0):
            print(f'Epoch {epoch} [{count}/{batches.size}] MSE: {value:.6f}')
    return total_loss / count

@torch.no_grad()
def predicting(model, batches):
    model.eval()
    labels, predictions = [], []
    for pair_indices, batch_labels in batches:
        output = model(pair_indices)
        predictions.append(output.cpu().reshape(-1))
        labels.append(batch_labels.cpu().reshape(-1))
    return torch.cat(labels).numpy(), torch.cat(predictions).numpy()









In [ ]:
"""reference split splits: five train/validation folds, one fixed test; ensemble summary in next cell.

Uses the original fold indices, not a new random KFold. Hyperparameters are supplied
by the caller; this entry point does not perform reference split's hyperparameter search.
"""
from __future__ import annotations

import csv
from dataclasses import dataclass
import gc
import hashlib
import json
from pathlib import Path
import time

import numpy as np
import torch


PROTOCOL = 'fixed_test_5fold_validation_v1'


@dataclass
class CVConfig(TrainingConfig):
    raw_data_dir: Path | None = None
    fold: int | None = None
    check_folds: bool = False

    def __post_init__(self):
        default_output = self.output_dir is None
        super().__post_init__()
        if default_output:
            self.output_dir = self.output_dir.with_name(self.output_dir.name + '_cv')
        self.raw_data_dir = Path(self.raw_data_dir) if self.raw_data_dir else ROOT / 'data'
        if self.fold is not None and self.fold not in range(5):
            raise ValueError('fold must be 0, 1, 2, 3 or 4')
        if self.resume_path is not None and self.fold is None:
            raise ValueError('--resume-path requires --fold; otherwise use automatic per-fold resume')


class PairSubset:
    """Only slice pair indices/labels; reuse the existing entity bank on every GPU."""
    def __init__(self, split, indices):
        indices = torch.as_tensor(indices, dtype=torch.long)
        if (indices.ndim != 1 or not len(indices) or indices.min() < 0
                or indices.max() >= len(split) or len(indices.unique()) != len(indices)):
            raise ValueError('Invalid or empty pair subset')
        self.dataset = split.dataset
        self.ligand_ids, self.protein_ids = split.ligand_ids, split.protein_ids
        self.pairs = {key: value.index_select(0, indices) if isinstance(value, torch.Tensor)
                      else [value[i] for i in indices.tolist()]
                      for key, value in split.pairs.items()}
        self.labels = self.pairs['y'].float().reshape(-1)

    __len__ = FourSourceSplit.__len__
    pair_ids = FourSourceSplit.pair_ids


def match_original_rows(split, original_rows):
    """Return original-order -> prepared-row mapping, checking identities AND labels."""
    if len(split) != len(original_rows):
        raise ValueError(f'{split.dataset}: prepared split has {len(split)} rows; '
                         f'reference split expects {len(original_rows)}')
    def matches(index, row):
        return (split.pair_ids(index) == (row['ligand_id'], row['protein_id'])
                and np.isclose(float(split.labels[index]), row['affinity'], rtol=0, atol=2e-6))

    # row_id is the position in the ORIGINAL FLATTENED SPLIT, not an index into Y.
    # Validate every identity/label before trusting it. This disambiguates KIBA
    # records whose distinct raw ligand IDs become the same canonical SMILES.
    try:
        positions = [int(value) for value in split.pairs['row_id']]
    except (TypeError, ValueError):
        positions = []
    if (sorted(positions) == list(range(len(split)))
            and all(matches(i, original_rows[pos]) for i, pos in enumerate(positions))):
        return np.argsort(positions).astype(np.int64)
    lookup = {}
    for i in range(len(split)):
        key = split.pair_ids(i)
        lookup.setdefault(key, []).append(i)
    result = []
    seen = set()
    for row in original_rows:
        key = (row['ligand_id'], row['protein_id'])
        candidates = [i for i in lookup.get(key, []) if i not in seen and matches(i, row)]
        if not candidates:
            raise ValueError(f'Pair missing/duplicated or affinity differs from reference split: {key}')
        if len(candidates) > 1:
            raise ValueError('Ambiguous duplicate reference split records: preserve original flattened-split '
                             'row_id when reordering prepared data')
        i = candidates[0]
        seen.add(i)
        result.append(i)
    return np.asarray(result, dtype=np.int64)




def regression_metrics(labels, predictions):
    """MSE/RMSE/Pearson and exact CI in O(n log n), excluding equal-label pairs."""
    y, p = np.asarray(labels, dtype=np.float64), np.asarray(predictions, dtype=np.float64)
    if (y.ndim != 1 or p.shape != y.shape or not len(y)
            or not np.isfinite(y).all() or not np.isfinite(p).all()):
        raise ValueError('Metrics require matching, non-empty, finite prediction/label vectors')
    mse = float(np.mean((y - p) ** 2))
    ranks = np.searchsorted(np.unique(p), p) + 1
    tree = np.zeros(int(ranks.max()) + 1, dtype=np.int64)

    def count(rank):
        total = 0
        while rank:
            total += int(tree[rank])
            rank -= rank & -rank
        return total

    order = np.argsort(y, kind='stable')
    concordant, comparable, previous, start = 0.0, 0, 0, 0
    while start < len(order):
        end = start + 1
        while end < len(order) and y[order[end]] == y[order[start]]:
            end += 1
        for index in order[start:end]:
            rank = int(ranks[index])
            below = count(rank - 1)
            concordant += below + 0.5 * (count(rank) - below)
            comparable += previous
        for index in order[start:end]:
            rank = int(ranks[index])
            while rank < len(tree):
                tree[rank] += 1
                rank += rank & -rank
        previous += end - start
        start = end
    pearson = float(np.corrcoef(y, p)[0, 1]) if np.ptp(y) > 0 and np.ptp(p) > 0 else None
    return {'mse': mse, 'rmse': mse ** 0.5,
            'ci': concordant / comparable if comparable else None, 'pearson': pearson}


def write_json(path, value):
    temporary = Path(path).with_suffix('.json.tmp')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + '\n')
    temporary.replace(path)


def run_fold(config, fold_id, train_data, valid_data, test_data, store, experiment_id):
    output = config.output_dir / f'fold_{fold_id}'
    output.mkdir(parents=True, exist_ok=True)
    seed_everything(config.seed)
    train_batches = IndexBatches(train_data, store, config.batch_size, config.shuffle, config.seed)
    valid_batches = IndexBatches(valid_data, store, config.test_batch_size)
    test_batches = IndexBatches(test_data, store, config.test_batch_size)
    model = configure_model(config.model_config, store)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    identity = {'protocol': PROTOCOL, 'fold': fold_id, 'experiment_id': experiment_id,
                'model_config': dict(config.model_config), 'model_signature': model_signature(model),
                'dataset_signature': dataset_signature((train_data, valid_data, test_data))}
    best, history, start_epoch = None, [], 1
    resume_path, resume = (find_resume_checkpoint([output, config.input_dir], identity, config.resume_path)
                           if config.auto_resume or config.resume_path else (None, None))
    if resume is not None:
        unwrap_model(model).load_state_dict(resume['model_state_dict'])
        optimizer.load_state_dict(resume['optimizer_state_dict'])
        best = cpu_snapshot(resume['best_checkpoint'])
        history, start_epoch = list(resume['history']), resume['epoch'] + 1
        restore_rng(resume['rng_state'], train_batches, store.devices)
        print(f'Fold {fold_id}: resume {resume_path}, next epoch {start_epoch}')
        del resume
    print(f'Fold {fold_id}: {len(train_data):,} train / {len(valid_data):,} validation / '
          f'{len(test_data):,} fixed test; checkpoint selection = validation MSE')
    with (output / 'history.csv').open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['epoch', 'train_mse', 'validation_mse', 'seconds'])
        writer.writeheader()
        writer.writerows(history)
        handle.flush()
        for epoch in range(start_epoch, config.epochs + 1):
            started = time.perf_counter()
            train_mse = train(model, train_batches, optimizer, epoch, config.log_interval)
            labels, predictions = predicting(model, valid_batches)
            if not np.isfinite(predictions).all():
                raise FloatingPointError('Validation prediction contains NaN/Inf')
            validation_mse = float(np.mean((labels.astype(np.float64) - predictions.astype(np.float64)) ** 2))
            if best is None or validation_mse < best['validation_mse']:
                best = {'model_type': MODEL_TYPE, **identity,
                        'model_state_dict': cpu_snapshot(unwrap_model(model).state_dict()),
                        'epoch': epoch, 'validation_mse': validation_mse,
                        'selection_split': 'validation', 'dataset': config.dataset}
                save_checkpoint(best, output / 'best_model.pt')
            record = {'epoch': epoch, 'train_mse': train_mse, 'validation_mse': validation_mse,
                      'seconds': time.perf_counter() - started}
            history.append(record)
            writer.writerow(record)
            handle.flush()
            if epoch % config.checkpoint_every == 0 or epoch == config.epochs:
                save_checkpoint({
                    'checkpoint_kind': 'training_resume_v1', **identity, 'epoch': epoch,
                    'model_state_dict': cpu_snapshot(unwrap_model(model).state_dict()),
                    'optimizer_state_dict': cpu_snapshot(optimizer.state_dict()),
                    'best_checkpoint': best, 'best_predictions': None,
                    'history': history, 'rng_state': capture_rng(train_batches, store.devices),
                }, output / f'training_checkpoint_epoch_{epoch:04d}.pt')
            print(f'Fold {fold_id}, epoch {epoch}: train={train_mse:.6f}, '
                  f'validation={validation_mse:.6f}, best epoch={best["epoch"]}')
    if best is None:
        raise RuntimeError('No trained checkpoint available')
    # Evaluate the held-out test only after checkpoint selection is finished.
    unwrap_model(model).load_state_dict(best['model_state_dict'])
    save_checkpoint(best, output / 'best_model.pt')
    labels, predictions = predicting(model, test_batches)
    metrics = regression_metrics(labels, predictions)
    save_test_predictions(output / 'test_predictions.csv', test_data, labels, predictions)
    result = {'protocol': PROTOCOL, 'experiment_id': experiment_id, 'fold': fold_id,
              'requested_epochs': config.epochs, 'completed_epoch': history[-1]['epoch'],
              'best_epoch': best['epoch'], 'validation_mse': best['validation_mse'],
              'n_train': len(train_data), 'n_validation': len(valid_data), 'n_test': len(test_data),
              'test_metrics': metrics}
    write_json(output / 'metrics.json', result)
    print(f'Fold {fold_id} test metrics: {metrics}')
    return result




def run_cross_validation(config: CVConfig):
    seed_everything(config.seed)
    if config.check_folds:
        train_data = FourSourceSplit(config.data_dir / 'train.pt', 'train', config.dataset)
        test_data = FourSourceSplit(config.data_dir / 'test.pt', 'test', config.dataset)
        groups = original_folds(train_data, test_data, config.raw_data_dir)
        result = {'dataset': config.dataset, 'validation_sizes': [len(g) for g in groups],
                  'training_sizes': [len(train_data) - len(g) for g in groups], 'test_size': len(test_data)}
        print(json.dumps(result, indent=2))
        return result
    device = torch.device(config.device if torch.cuda.is_available() else 'cpu')
    train_data, test_data, store, unused_train, unused_test = load_data(
        config.data_dir, config.dataset, device, config.batch_size, config.test_batch_size,
        config.max_gpus, config.shuffle, config.seed)
    del unused_train, unused_test
    groups = original_folds(train_data, test_data, config.raw_data_dir)
    probe_model = configure_model(config.model_config, store)
    architecture = model_signature(probe_model)
    del probe_model
    experiment = {'protocol': PROTOCOL, 'dataset': config.dataset,
                  'dataset_signature': dataset_signature((train_data, test_data)),
                  'fold_signature': hashlib.sha256(b''.join(g.tobytes() for g in groups)).hexdigest(),
                  'model_config': dict(config.model_config), 'model_signature': architecture,
                  'optimizer': 'Adam', 'lr': config.lr,
                  'batch_size': config.batch_size, 'test_batch_size': config.test_batch_size,
                  'shuffle': config.shuffle, 'seed': config.seed}
    experiment_id = hashlib.sha256(json.dumps(experiment, sort_keys=True).encode()).hexdigest()
    config.output_dir.mkdir(parents=True, exist_ok=True)
    manifest = config.output_dir / 'experiment.json'
    if manifest.exists() and json.loads(manifest.read_text()) != experiment:
        raise ValueError('Output belongs to a different CV experiment; choose a new --output-dir')
    write_json(manifest, experiment)
    with (config.output_dir / 'fold_assignments.csv').open('w', newline='') as handle:
        writer = csv.writer(handle)
        writer.writerow(['prepared_train_index', 'row_id', 'validation_fold', 'ligand_id', 'protein_id'])
        for fold, indices in enumerate(groups):
            for i in indices:
                writer.writerow([int(i), train_data.pairs['row_id'][i], fold, *train_data.pair_ids(i)])
    for fold in range(5) if config.fold is None else [config.fold]:
        training_indices = np.concatenate([g for i, g in enumerate(groups) if i != fold])
        run_fold(config, fold, PairSubset(train_data, training_indices),
                 PairSubset(train_data, groups[fold]), test_data, store, experiment_id)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        summary = summarize(config.output_dir, experiment_id, config.epochs)
        print(f'Completed {len(summary["completed_folds"])}/5 folds. Summary: '
              f'{config.output_dir / "cv_summary.json"}')
    return summary







def original_folds(train_data, test_data, raw_data_dir=None):
    """Restore original interaction folds; verify pairs AND labels in both .pt files."""
    if test_data.dataset != train_data.dataset:
        raise ValueError('Train/test must belong to the same dataset')
    name = train_data.dataset
    if FOLDS_PATH is not None:
        path = Path(FOLDS_PATH)
    else:
        filename = f'{name}_folds.json'
        path = Path(DATA_DIR) / filename
        if not path.is_file():
            if Path('/kaggle/input').is_dir():
                candidates = sorted(Path('/kaggle/input').rglob(filename))
            else:
                candidates = [REPO_ROOT / 'data/cv_folds' / filename]
                candidates = [p for p in candidates if p.is_file()]
            if len(candidates) != 1:
                raise ValueError(f'Found {len(candidates)} files named {filename}; set FOLDS_PATH explicitly.')
            path = candidates[0]
    if not path.is_file():
        raise FileNotFoundError(f'Fold file not found: {path}; update FOLDS_PATH.')
    payload = json.loads(path.read_text())
    if payload.get('format_version') != 1 or payload.get('dataset') != name:
        raise ValueError(f'{path}: incorrect dataset or fold file format')
    def expected_rows(split):
        return [{'ligand_id': payload['ligand_ids'][li],
                 'protein_id': payload['protein_ids'][pi], 'affinity': y}
                for li, pi, y in payload[split]]
    train_order = match_original_rows(train_data, expected_rows('train'))
    match_original_rows(test_data, expected_rows('test'))
    sizes = payload['fold_sizes']
    if (len(sizes) != 5 or any(type(n) is not int or n < 1 for n in sizes)
            or sum(sizes) != len(train_order)):
        raise ValueError('Invalid fold sizes')
    print('Fold file:', path)
    return list(np.split(train_order, np.cumsum(sizes)[:-1]))


In [ ]:
import csv


import gc


import json


from pathlib import Path


import numpy as np


import torch


AGGREGATION = 'arithmetic_mean_of_5_fold_predictions_then_metrics_v1'


def ensemble_metrics(labels, predictions_by_fold):
    if set(predictions_by_fold) != set(range(5)):
        raise ValueError('Ensemble requires exactly 5 folds with distinct IDs: 0, 1, 2, 3, 4')
    labels = np.asarray(labels, dtype=np.float64)
    predictions = [np.asarray(predictions_by_fold[i], dtype=np.float64) for i in range(5)]
    # Validate every model before averaging: never silently drop NaN/missing folds.
    individual = {str(i): regression_metrics(labels, p) for i, p in enumerate(predictions)}
    matrix = np.stack(predictions, axis=1)
    mean_prediction = matrix.mean(axis=1)
    return matrix, mean_prediction, regression_metrics(labels, mean_prediction), individual


def export_ensemble(output_dir, rows, predictions_by_fold, metadata):
    """Rows and every prediction vector must refer to the same ordered test samples."""
    labels = np.asarray([float(row['affinity']) for row in rows], dtype=np.float64)
    matrix, mean_prediction, metrics, individual = ensemble_metrics(labels, predictions_by_fold)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    prediction_path = output_dir / 'ensemble_predictions.csv'
    temporary = prediction_path.with_suffix('.csv.tmp')
    with temporary.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.writer(handle)
        writer.writerow(['row_id', 'ligand_id', 'protein_id', 'y_true',
                         *[f'y_pred_fold_{i}' for i in range(5)], 'y_pred_mean'])
        for index, row in enumerate(rows):
            writer.writerow([row['row_id'], row['ligand_id'], row['protein_id'],
                             labels[index], *matrix[index], mean_prediction[index]])
    temporary.replace(prediction_path)
    summary = {**metadata, 'aggregation': AGGREGATION, 'complete': True,
               'completed_folds': list(range(5)), 'n_test': len(labels),
               'test_metrics': metrics, 'per_model_test_metrics': individual,
               'prediction_file': prediction_path.name}
    write_json(output_dir / 'ensemble_metrics.json', summary)
    with (output_dir / 'ensemble_metrics.csv').open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['model', *metrics])
        writer.writeheader()
        for fold, values in individual.items():
            writer.writerow({'model': f'fold_{fold}', **values})
        writer.writerow({'model': 'ensemble_mean_prediction', **metrics})
    return summary


def read_prediction_rows(path):
    with Path(path).open(newline='', encoding='utf-8') as handle:
        reader = csv.DictReader(handle)
        required = {'row_id', 'ligand_id', 'protein_id', 'affinity', 'predicted_affinity'}
        if not required.issubset(reader.fieldnames or []):
            raise ValueError(f'{path}: missing ID/affinity/predicted_affinity columns')
        rows = list(reader)
    keys = [(r['row_id'], r['ligand_id'], r['protein_id']) for r in rows]
    if not rows or len(set(keys)) != len(keys):
        raise ValueError(f'{path}: empty test set or duplicate (row_id, ligand_id, protein_id) keys')
    return rows, keys


def summarize(output_dir, experiment_id, epochs):
    """Replacement for the original notebook's metric-mean summary; retain CV training."""
    output_dir = Path(output_dir)
    records = []
    for fold in range(5):
        path = output_dir / f'fold_{fold}/metrics.json'
        if path.is_file():
            record = json.loads(path.read_text())
            if (record.get('experiment_id') == experiment_id and record.get('fold') == fold
                    and record.get('requested_epochs') == epochs
                    and record.get('completed_epoch', 0) >= epochs):
                records.append(record)
    summary = {'experiment_id': experiment_id, 'requested_epochs': epochs,
               'aggregation': AGGREGATION, 'complete': False,
               'completed_folds': [r['fold'] for r in records], 'test_metrics': None,
               'folds': records}
    # Invalidate an old complete export before attempting a new aggregation.
    for filename in ('ensemble_predictions.csv', 'ensemble_metrics.json', 'ensemble_metrics.csv'):
        (output_dir / filename).unlink(missing_ok=True)
    write_json(output_dir / 'cv_summary.json', summary)
    if len(records) < 5:
        print(f'Ensemble pending: all 5 folds are required, but only {len(records)} are available. No partial average.')
        return summary
    reference_rows, reference_keys = read_prediction_rows(output_dir / 'fold_0/test_predictions.csv')
    reference_labels = np.asarray([float(r['affinity']) for r in reference_rows])
    predictions = {}
    for record in records:
        fold = record['fold']
        rows, keys = read_prediction_rows(output_dir / f'fold_{fold}/test_predictions.csv')
        if set(keys) != set(reference_keys):
            raise ValueError(f'Fold {fold}: test sample IDs differ from fold 0')
        by_key = dict(zip(keys, rows))
        ordered = [by_key[key] for key in reference_keys]
        labels = np.asarray([float(row['affinity']) for row in ordered])
        if not np.array_equal(labels, reference_labels):
            raise ValueError(f'Fold {fold}: test labels differ from fold 0')
        predictions[fold] = np.asarray([float(row['predicted_affinity']) for row in ordered])
        calculated = regression_metrics(labels, predictions[fold])
        for key, value in calculated.items():
            saved = record['test_metrics'].get(key)
            if ((value is None) != (saved is None)
                    or value is not None and not np.isclose(value, saved, rtol=1e-10, atol=1e-12)):
                raise ValueError(f'Fold {fold}: CSV does not match metrics.json ({key})')
        if record.get('n_test') != len(ordered):
            raise ValueError(f'Fold {fold}: test sample count does not match metrics.json')
    summary = export_ensemble(output_dir, reference_rows, predictions, summary)
    write_json(output_dir / 'cv_summary.json', summary)
    return summary




## Running and resuming

`FOLD=None`: run all five folds sequentially. `FOLD=0..4`: train one fold; aggregate only when
the current output directory contains five compatible results completed through NUM_EPOCHS.
`CHECK_FOLDS_ONLY=True`: validate splits on CPU only.

Resuming restores optimizer/RNG/history and requires the same protocol and experiment.
To combine folds across sessions,
add previous outputs as input and run FOLD=None: completed folds are loaded and reevaluated;
incomplete folds resume training. `best_model.pt` supports inference, not resuming.

Aggregate only predictions from each fold's best validation-selected checkpoint.
CSV rows are matched by (row_id, ligand_id, protein_id) and labels rather than relying
on row order. Different experiments and unfinished training runs cannot be mixed.
To evaluate five existing models only, use `four-source-ensemble-predict.ipynb`.


In [ ]:
config = CVConfig(
    dataset=DATASET, data_dir=DATA_DIR, output_dir=OUTPUT_DIR, input_dir=INPUT_DIR,
    fold=FOLD, check_folds=CHECK_FOLDS_ONLY,
    device=CUDA_DEVICE, max_gpus=MAX_GPUS,
    batch_size=TRAIN_BATCH_SIZE, test_batch_size=TEST_BATCH_SIZE,
    lr=LR, epochs=NUM_EPOCHS, checkpoint_every=CHECKPOINT_EVERY,
    auto_resume=AUTO_RESUME, resume_path=RESUME_PATH,
    seed=SEED, shuffle=SHUFFLE_TRAIN, log_interval=LOG_INTERVAL,
    model_config=MODEL_CONFIG)
result = run_cross_validation(config)


In [ ]:
if CHECK_FOLDS_ONLY:
    print('Validated folds:', result)
else:
    print('Completed:', len(result['completed_folds']), '/ 5 folds')
    if result['complete']:
        print('Metrics from the mean prediction of five models:')
        for metric, value in result['test_metrics'].items():
            print(f'{metric}: {value}')
        print('Prediction:', OUTPUT_DIR / 'ensemble_predictions.csv')
        print('Ensemble and per-fold metrics:', OUTPUT_DIR / 'ensemble_metrics.csv')
    else:
        print('Ensemble pending: fewer than five folds. Continue the remaining folds.')
